# Notebook 09 — Deep energy encoder: local and FMM hierarchy ablations

**Run All launches the actual pilot in detached tmux.** This build has not launched the pilot. Expected runtime is approximately **2–3 hours**, subject to actual calibration; an independent watcher enforces **three hours from first launch**, including all preparation, recovery checks, training and evaluation. Run All returns after launch, and the worker continues when the browser disconnects.

The question is whether adding learned nonlinear depth makes energy interactions useful within JEPA, and whether the FMM hierarchy helps beyond local interactions. All results are a **two-seed exploratory pilot**, conditional on the reused Food101 development set. There is no confirmatory significance claim, winner-based stopping, or automatic larger run.

## Fixed experiment

| Architecture | Four feed-forward blocks | Token interactions |
|---|---|---|
| ViT | Yes | Self-attention |
| MLP | Yes | None |
| Local energy | Yes | Local energy |
| Local + FMM energy | Yes | Local and hierarchical energy |

The shared backbone uses 128px images, 8px patches, width 192, four independently learned blocks, identical 4× expansion feed-forward networks, residual connections and normalization. All models share initial patch stem, corresponding feed-forward/norm parameters and the existing width-128 two-block JEPA predictor, with independent tensor storage. Parameter counts remain different and are reported; unused parameters are not presented as active capacity.

**Eight training runs:** seeds `20261201` and `20261202`, all four architectures, exactly **350 successful updates per model**, physical/effective batch 128, FP32, and 44,800 image exposures per model. Paired models receive identical images, augmentations and masks; these random streams are fresh. Prior pilots and Notebook 08 results are excluded.

**Sixteen fixed evaluations:** each initialized and trained EMA encoder gets an epoch-20 frozen-feature linear probe with training-only standardization and 800 probe updates. The ordinary JEPA loss and EMA recipe are shared; pretraining does not use class labels. Step timing includes the complete JEPA training step: context and target encoders, predictor, backward pass, optimizer and EMA. Classification accuracy measures the frozen encoder after pretraining. The predictor remains a Transformer.

Registered architectural comparisons for this exploratory pilot are local minus MLP, local+FMM minus local, and local+FMM minus ViT. Report both individual seed pairs, own-initialization gains, actual training time, active/total parameters and allocated memory. Two-seed intervals are fragile and do not establish a general superiority claim. This experiment matches exposure; matching time would require a separately registered experiment.


## Energy block definition

For each block, set $u=\operatorname{LN}(h_\ell)$ and construct a separate conditional scalar energy:

$$E_\ell(z\mid u)=\tfrac12\|z-u\|^2+\tfrac{\lambda_n}{2}\|B_{\ell,n}(u)z-b_{\ell,n}(u)\|^2+\tfrac{\lambda_f}{2}\|B_{\ell,f}(u)z-b_{\ell,f}(u)\|^2.$$

The coefficients are fixed within the inner solve and independently learned per block. Start at $z_0=u$, take one descent step with $\eta=.5$, then use

$$\tilde h=h_\ell+W_{\ell,o}(z_1-u),\qquad h_{\ell+1}=\tilde h+\operatorname{MLP}_\ell(\operatorname{LN}(\tilde h)).$$

For local energy, $\lambda_n=.25$ and the far term is absent. Local+FMM uses $\lambda_n=\lambda_f=.25$ and the existing width-32 projected cell-mean interactions. There are no inactive far-energy parameters in the local control. Both online and EMA target encoders use the selected architecture.

At the default one-step initialization $z_0=u$, the anchor gradient is zero initially: the block applies the local/far energy-gradient correction. The anchor directly affects later inner steps when using two or more.

The conditional energy step has the inherited fixed-conditioning descent guarantee. The four-block network changes its conditioning between blocks; it is not inference in one global scalar energy, and the old four-step contraction bound does not describe this complete network. Better semantic representations remain an experimental hypothesis.


The registered execution recipe retains activations for backward and processes 1,024 edges per tile. It preserves the energy and architecture; output/gradient equivalence is checked to floating-point tolerance. Full-state recovery checkpoints use the separate schedule below.


## Inputs, outputs and exact recovery

The fixed mask, optimizer and predictor recipe is read from `artifacts/notebook_07/energy-encoder-001/stages/learning/plan.json` and hash-bound; Notebook 07/08 export definitions are dependencies.

Use the existing sealed Food101 **10,100 training/probe-training images** and **2,020 development images**. Official test data remains reserved. A separate software-validation probe may reuse hash-verified old feature caches as fixtures; those scores never enter this study's outcomes.

Outputs: `artifacts/notebook_09/deep-energy-pilot-001/`. Final report and plots: `stages/learning/stage_results.md`. Checkpoints: `checkpoints/deep-energy-pilot-001/`. Save at each epoch/stage boundary and every 50 optimizer updates or 15 minutes, retaining at least the latest two valid resumable checkpoints and the best. Manifests verify atomic writes and store model, optimizer, scheduler, scaler/precision, every RNG, sample/mask stream, data cursor, global step, configuration and metrics. One writer per run.

Before the long comparison, the worker runs discarded calibration, verifies bitwise full-state 4-step versus interrupted 2+2-step recovery for every architecture, and validates actual encoder feature-cache/probe integration. Calibration checks feasibility only. If the complete eight-run/16-probe study does not fit the original cap, it stops explicitly without reducing the study.

**Exact resume:** select the `ahr-go` kernel and Run All unchanged with the same RUN_ID before the original deadline. It reconnects to the existing tmux worker or resumes verified checkpoints. The original clock never resets. Invalid bindings/checkpoints fail explicitly; a cutoff is INCOMPLETE and retains valid state. Completed results are reused. Changing code or architecture requires a new registered run.

Attach: `tmux attach -t fmm-deep-energy-deep-energy-pilot-001`. Detach: Ctrl-b, then d. The deadline watcher targets only this named study. Storage is on the persistent `/dev/sda1` boot disk; instance-deletion policy remains unverified. Final/best checkpoints must be copied off-instance and hashes verified before deleting or recycling the machine.


## 1. Load tagged definitions

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import nbformat, importlib.util, sys
from IPython.display import display, Markdown, Image
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa").resolve()
_boot = {}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)
_spec = importlib.util.spec_from_loader("fmm09_workflow", _boot["TaggedNotebookLoader"](ROOT / "experiments/09_deep_energy/workflow.ipynb"))
WORKFLOW = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = WORKFLOW
_spec.loader.exec_module(WORKFLOW)


## 2. Registered configuration
The model variants, seeds, update/probe budgets and three-hour cap are fixed. Inspect them here before launch.

In [ ]:
RUN_ID = "deep-energy-pilot-001"
CONFIG = WORKFLOW.study_configuration(RUN_ID)
CONFIG


## 3. Check readiness and model sizes
This check creates no experiment clock, checkpoints or worker. It verifies the saved build-validation certificates and reports actual architecture counts. Scientific results remain NOT_RUN until the experiment completes.

In [ ]:
READINESS = WORKFLOW.readiness(ROOT, CONFIG)
import pandas as pd
display(pd.DataFrame(READINESS["architecture_summary"]))
display({key: READINESS[key] for key in ("status", "production_run_started", "expected_runtime", "planned_training_runs", "planned_probe_evaluations", "checkpoint_location", "scientific_results")})


## 4. Launch or reconnect
**This cell starts the real experiment.** Run All executes it automatically. The saved binding fixes the original clock; the detached worker performs calibration and recovery checks before training.

In [ ]:
LAUNCH = WORKFLOW.launch_study(ROOT, CONFIG)
DISPLAY_LAUNCH = {key: value for key, value in LAUNCH.items() if key != "command"}
if LAUNCH.get("deadline_unix"):
    DISPLAY_LAUNCH["hard_stop_dublin"] = datetime.fromtimestamp(LAUNCH["deadline_unix"], ZoneInfo("Europe/Dublin")).isoformat()
display(DISPLAY_LAUNCH)


## 5. Inspect progress
Rerun this cell for stage status, saved update counts and time remaining. Accuracy comparisons appear after all training runs and evaluations complete.

In [ ]:
STATUS = WORKFLOW.inspect_study(ROOT, RUN_ID)
PROGRESS = STATUS.get("saved_training_progress", [])
display({key: value for key, value in STATUS.items() if key != "saved_training_progress"})
if PROGRESS:
    display(pd.DataFrame(PROGRESS))


## 6. Read completed results
Only real completed experiment results are rendered here. A partial run is not analyzed as the planned full study.

In [ ]:
REPORT_FOLDER = ROOT / "artifacts/notebook_09" / RUN_ID / "stages/learning"
REPORT = REPORT_FOLDER / "stage_results.md"
if STATUS.get("status") == "COMPLETED" and REPORT.exists():
    display(Markdown("\n".join(line for line in REPORT.read_text().splitlines() if not line.startswith("!["))))
    for figure in ("stage_quality.png", "stage_compute.png", "stage_contrasts.png"):
        figure_path = REPORT_FOLDER / figure
        if figure_path.exists():
            display(Image(filename=str(figure_path)))
else:
    display(Markdown("**No complete scientific result yet.** Use the progress cell above; partial or validation-only scores are not pilot conclusions."))


## Interpretation after completion

Assess own-initialization learning, local minus MLP, and local+FMM minus local using both seed pairs. Compare the measured cost of each architecture. The global energy model earns further investigation only if its learning improvement and incremental hierarchy benefit justify its cost. A promising pilot would need separate confirmation with more fresh seeds. This notebook does not automatically select, publish or start a larger run.

Method references: [I-JEPA paper](https://arxiv.org/abs/2301.08243) and [official implementation](https://github.com/facebookresearch/ijepa). Model, mask and optimization recipes here are a controlled small custom implementation; this is not a reproduction of published large-scale scores.
